# 03 — Association Rule Modelling & Algorithm Benchmarking
**Group 6 | Topic 6: Market Basket Analysis on Online Retail II | `SEED = 5018`**

This notebook benchmarks **Apriori** vs. **FP-Growth** using `mlxtend` across support and confidence thresholds and inspects top rules ranked by lift, confidence, support, and conviction.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import GROUP_SEED
from src.association_rules import format_rules_table, run_apriori, run_fpgrowth
from src.download_data import download_dataset
from src.preprocessing import create_basket_matrix, load_and_sample_data

SEED = GROUP_SEED  # 5018
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

## 1. Prepare Sampled Basket Matrix (`SEED = 5018`)

In [ ]:
dataset_path = download_dataset(data_dir=PROJECT_ROOT / "data")
sampled_df = load_and_sample_data(dataset_path, seed=SEED, n_invoices=5000)
basket_matrix = create_basket_matrix(sampled_df, sparse=True)
print(f"Basket matrix ready: {basket_matrix.shape}")

## 2. Apriori vs. FP-Growth Runtime & Memory Benchmark (RQ1 & RQ2)

In [ ]:
support_grid = [0.03, 0.02, 0.015, 0.01]
confidence_grid = [0.30, 0.50, 0.70]

benchmark_records = []
for min_support in support_grid:
    for min_confidence in confidence_grid:
        apriori_res = run_apriori(
            basket_matrix,
            min_support=min_support,
            min_threshold=min_confidence,
        )
        fpgrowth_res = run_fpgrowth(
            basket_matrix,
            min_support=min_support,
            min_threshold=min_confidence,
        )
        benchmark_records.append(apriori_res["metrics"])
        benchmark_records.append(fpgrowth_res["metrics"])

benchmark_df = pd.DataFrame(benchmark_records)
display(benchmark_df)

## 3. High-Lift Cross-Selling Associations & Evaluation Metrics (RQ3)

In [ ]:
ref_result = run_fpgrowth(basket_matrix, min_support=0.015, min_threshold=0.50)
rules_df = ref_result["rules"]

if not rules_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    scatter = ax.scatter(
        rules_df["support"],
        rules_df["confidence"],
        c=rules_df["lift"],
        cmap="viridis",
        alpha=0.8,
    )
    fig.colorbar(scatter, ax=ax, label="Lift")
    ax.set_title("Association Rules: Support vs. Confidence (Colored by Lift)")
    ax.set_xlabel("Support")
    ax.set_ylabel("Confidence")
    plt.tight_layout()
    plt.show()

display(format_rules_table(rules_df, top_n=15))